In [77]:
import sys

sys.path.append('../')


In [78]:
from device import get_device, clear_gpu_full
device = get_device()
print(f"Using device: {device}")

Using device: cuda


In [79]:
import torch
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [80]:
SYSTEM_PROMPT = """You are a text rewriting and generation engine for a scientific benchmark.
Return only the requested text.
Do not add explanations, comments, markdown, headings, or prefaces."""

In [81]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_model_bf16(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    model.eval()
    return tokenizer, model



def load_model_4bit_nf4(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model.eval()
    return tokenizer, model

    


In [82]:
import torch

@torch.inference_mode()
def generate_text(
    tokenizer,
    model,
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 40,
    max_new_tokens: int = 256,
):
        # Gemma chat template does not support sy"stem role.
    model_id = model.config._name_or_path
    if "gemma" in model_id.lower():
        print("GEMMA MODEL")
        merged_prompt = f"""Instruction:
{system_prompt}

Task:
{user_prompt}

Return only the requested output."""
        messages = [
            {"role": "user", "content": merged_prompt}
        ]


    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][input_length:]

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [83]:
# ok
# model_id = "meta-llama/Llama-3.2-3B-Instruct"
# model_id = "mistralai/Mistral-7B-Instruct-v0.3"
model_id = "google/gemma-2-9b-it"
# model_id = "Qwen/Qwen2.5-7B-Instruct"
# 3 test

tokenizer, model = load_model_bf16(model_id)

Loading weights: 100%|██████████| 464/464 [00:00<00:00, 802.92it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [84]:
output = generate_text(
    tokenizer=tokenizer,
    model=model,
    user_prompt="Hi there! Can you help me with a scientific question?",
)


GEMMA MODEL


In [85]:
print(output)

I'd love to try! Ask away.


In [86]:
del(output)
del(model)
del(tokenizer)
clear_gpu_full()

GPU cleanup attempted
allocated: 16501.19287109375 MB
reserved: 17664.0 MB
